# Deploying Transformers Models on Databricks Model Serving
In this example we explore how you can deploy and inference upon Transformers models utilizing Databricks Model Serving. We use a smaller CPU based BERT model here, for GPU workloads ensure you have that enabled for your workspace.

### Additional Resources/References
- Model Serving Docs: https://docs.databricks.com/aws/en/machine-learning/model-serving/
- Official NB Examples: https://docs.databricks.com/aws/en/machine-learning/model-serving/create-manage-serving-endpoints#notebook-examples

## Unity Catalog Setup

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS custom_ml
MANAGED LOCATION 'Enter metastore artifact path';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS custom_ml.models;

## MLflow Setup
Enabling signature tracing and creating experiment/run to register the model in Unity Catalog.

In [0]:
import mlflow
import pandas as pd
from transformers import pipeline
from mlflow.models.signature import infer_signature

In [0]:
# 1) Build HF pipeline
# Device param (toggle for CPU & GPU): https://huggingface.co/docs/transformers/en/pipeline_tutorial#device
hf_pipe = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    #device=0,
    truncation=True
)

# 2) Prepare an input example + signature (recommended; required for UC logging in many setups)
input_example = pd.DataFrame({"text": ["Databricks serving is great."]})
example_output = hf_pipe(input_example["text"].tolist())
print(example_output)
signature = infer_signature(input_example, example_output)
print("----------")
print("Input and output shapes")
print("----------")
print(signature)

In [0]:
# 3) Ensure we use Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")

# Catalog details
catalog = "custom_ml"
schema = "models" 
model_name = "hf_bert_transformers"
# Adjust model name as needed
registered_name = f"{catalog}.{schema}.{model_name}"

## Track Experiment/Run & Register Model

In [0]:
# 4) Log and register with MLflow Transformers flavor: https://mlflow.org/docs/latest/ml/deep-learning/transformers/

# Track experiment & run
with mlflow.start_run(run_name="hf_transformers_bert"):
    mlflow.transformers.log_model(
        transformers_model=hf_pipe,
        artifact_path="model",
        input_example=input_example,
        signature=signature,
    )
    run_id = mlflow.active_run().info.run_id

model_uri = f"runs:/{run_id}/model"
print(model_uri)

# Register explicitly in UC, issues with autolog at times depending on package version
res = mlflow.register_model(model_uri=model_uri, name=registered_name)
print("Model URI:", model_uri)
print("Registered UC model:", registered_name, "version:", res.version)

## Create Endpoint

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client

#mlflow.set_registry_uri("databricks-uc")
client = get_deploy_client("databricks") #Client to interfact with deployment & inference

# specify version
version = "1" 

# Main Deployment API
endpoint = client.create_endpoint(
    name="bert-transformers-ep",
    config={
        "served_entities": [
            {
                "name": f"{model_name}-{version}",                 # served model name on the endpoint
                "entity_name": f"{catalog}.{schema}.{model_name}", # UC model
                "entity_version": version,
                "workload_size": "Small",
                "workload_type": "CPU",
                "scale_to_zero_enabled": True
            }
        ]
    }
)

In [0]:
import time
endpoint_name = "bert-transformers-ep"

while True:
    ep = client.get_endpoint(endpoint_name)
    state = ep["state"]["ready"]

    print(f"Endpoint state: {state}")

    if state == "READY":
        print("✅ Endpoint is ready")
        break

    if state == "FAILED":
        raise RuntimeError(f"❌ Endpoint creation failed: {ep}")

    time.sleep(45)  # poll every 45s

## Sample Inference

In [0]:
# sample payload, shape: 
payload = {
  "dataframe_split": {
    "columns": ["text"],
    "data": [["I am so angry and upset."]]
  }
}
print(payload)
# Invoke the endpoint with your raw payload
resp = client.predict(
    endpoint=endpoint_name,
    inputs=payload,
)
print(resp)

In [0]:
import random

# helper to structure payload
def structure_payload(samp_inp: str) -> dict:
    return {
        "dataframe_split": {
            "columns": ["text"],
            "data": [[samp_inp]]
        }
    }

# randomly choose from payloads for a larger test
sample_payloads = [
    "I am sooooooooo happy",
    "I am ANGRY AND MAD AND FURIOUS",
    "Databricks serving is great"
]

# iterate over sample payloads and do inference
for payload in sample_payloads:
    selected_payload = random.choice(sample_payloads)
    print(selected_payload)
    structured_payload = structure_payload(selected_payload)
    resp = client.predict(
        endpoint=endpoint_name,
        inputs=structured_payload,
    )
    print(resp)